# Chapter 4: Introduction to Gradient Descent — Learning to Reduce Error
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"The only relevant test of the validity of a hypothesis is comparison of its predictions with experience."*  
> — Milton Friedman (1953)

---

## 1. Executive Summary & Objective

Having established forward propagation in Chapter 3, we now complete the foundational loop of machine learning by exploring the remaining two stages: **Compare** and **Learn**.

### Core Objectives:
1. **Formalize Error Measurement**: Understand why Mean Squared Error (MSE) is chosen over raw difference or absolute error.
2. **Deconstruct 'Hot and Cold' Learning**: Implement trial-and-error weight perturbation and analyze its severe combinatorial and precision limitations.
3. **Derive Gradient Descent**: Mathematically derive how calculus directly reveals both the *direction* and *exact magnitude* to adjust parameters.
4. **Analyze the Role of Alpha ($\alpha$)**: Demystify the learning rate hyperparameter and observe real-world phenomena of underfitting, optimal convergence, and catastrophic overshooting (divergence).

---
## 2. Theoretical Deep-Dive

### 2.1 The Compare Step: Why Squared Error?

Given a prediction $\hat{y}$ and a ground-truth target $y$, why don't we simply use the raw difference $(\hat{y} - y)$ as error?

1. **Sign Cancellation**: If one prediction misses by $+10$ and another misses by $-10$, the average raw error is $\frac{10 + (-10)}{2} = 0$, falsely claiming a perfect model.
2. **The Squared Error Formulation**:

$$\text{Error} = (\hat{y} - y)^2$$

**Mathematical Properties of Squared Error:**
- **Non-negativity**: $\text{Error} \ge 0$ always. Whether you undershoot or overshoot, error is strictly positive.
- **Disproportionate Penalty**: Large errors are penalized heavily ($10^2 = 100$), forcing the optimizer to prioritize catastrophic mistakes over minor noise ($0.1^2 = 0.01$).
- **Smooth Convex Derivative**: The function forms a continuous upward parabola, ensuring clean, continuous derivatives everywhere.

### 2.2 From Heuristics to Calculus: Hot and Cold vs. Gradient Descent

#### Why 'Hot and Cold' Learning Fails
In Hot and Cold learning, we manually tweak a weight slightly up ($w + \epsilon$) and down ($w - \epsilon$), check the new error, and keep whichever modification decreased the error.

- **Combinatorial Explosion**: For $N$ weights, evaluating combinations of step changes requires $3^N$ forward evaluations per step. For a modest network with 100,000 weights, $3^{100000}$ exceeds the number of atoms in the known universe.
- **Step Size Inefficiency**: A fixed $\epsilon$ oscillates endlessly around the minimum without converging to the precise analytical minimum.

#### The Analytical Solution: Gradient Descent
Using differential calculus, we determine the slope (derivative) of the error with respect to the weight $w$:

$$\text{Error} = (w \cdot x - y)^2$$

Applying the Chain Rule:

$$\frac{\partial \text{Error}}{\partial w} = 2 \cdot (w \cdot x - y) \cdot \frac{\partial (w \cdot x - y)}{\partial w} = 2 \cdot (\hat{y} - y) \cdot x$$

In Andrew Trask's formulation (absorbing the constant 2 into the learning rate $\alpha$):

$$\text{delta} = \hat{y} - y = \text{pred} - \text{actual}$$
$$\text{weight\_delta} = \text{delta} \cdot \text{input}$$

**The Three Critical Properties of `weight_delta`:**
1. **Direction**: The sign of `delta` dictates whether the weight should increase or decrease.
2. **Amount**: The magnitude of `delta` automatically scales the step size (large error $\to$ large step; near zero $\to$ tiny step).
3. **Input Scaling & Reversal**:
   - If $\text{input} = 0$, $\text{weight\_delta} = 0$. (If a feature was absent, the weight cannot be held responsible for the error).
   - If $\text{input} < 0$, multiplying by a negative number inverts the direction correctly, preserving descent down the error bowl.

### 2.3 The Role of Alpha ($\alpha$)

The learning rate $\alpha$ controls how aggressively we adjust the weight based on the calculated gradient:

$$w \leftarrow w - \alpha \cdot \text{weight\_delta}$$

- **Optimal $\alpha$**: Smooth, exponential decay of error towards the global minimum.
- **Too Small $\alpha$**: Extremely slow progress requiring excessive iterations.
- **Too Large $\alpha$ (Overshooting)**: The update jumps over the parabola's minimum to an even higher point on the opposing slope, causing exponential divergence.

---
## 3. Code Reproduction

We now reproduce each experiment described in Chapter 4.

In [ ]:
# 3.1 Measuring Error (Mean Squared Error)
knob_weight = 0.5
input_data = 0.5
goal_prediction = 0.8

pred = input_data * knob_weight
error = (pred - goal_prediction) ** 2

print(f"Prediction:      {pred:.4f}")
print(f"Goal Target:     {goal_prediction:.4f}")
print(f"Calculated Error: {error:.4f}")

In [ ]:
# 3.2 'Hot and Cold' Learning Implementation

weight = 0.5
input_val = 0.5
goal_prediction = 0.8
step_amount = 0.001

print("Starting Hot & Cold Learning:")
for iteration in range(1101):
    prediction = input_val * weight
    error = (prediction - goal_prediction) ** 2
    
    # Try moving weight up
    up_prediction = input_val * (weight + step_amount)
    up_error = (goal_prediction - up_prediction) ** 2
    
    # Try moving weight down
    down_prediction = input_val * (weight - step_amount)
    down_error = (goal_prediction - down_prediction) ** 2
    
    if down_error < up_error:
        weight -= step_amount
    if down_error > up_error:
        weight += step_amount
        
    if iteration % 100 == 0:
        print(f"Iteration {iteration:4d} | Error: {error:8.6f} | Prediction: {prediction:6.4f} | Weight: {weight:6.4f}")

In [ ]:
# 3.3 Gradient Descent: Calculating Direction and Amount (One Step)

weight = 0.5
goal_pred = 0.8
input_val = 0.5

for iteration in range(1):
    pred = input_val * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = delta * input_val
    weight = weight - weight_delta
    print(f"Step 1 Output:")
    print(f"  Error:        {error:.6f}")
    print(f"  Prediction:   {pred:.6f}")
    print(f"  Delta:        {delta:.6f}")
    print(f"  Weight Delta: {weight_delta:.6f}")
    print(f"  Updated W:    {weight:.6f}")

In [ ]:
# 3.4 Multi-Iteration Gradient Descent: Watching Error Converge to 0

weight = 0.5
goal_pred = 0.8
input_val = 2.0
alpha = 0.1

print("Starting Gradient Descent Convergence:")
for iteration in range(20):
    pred = input_val * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = delta * input_val
    weight = weight - (alpha * weight_delta)
    
    print(f"Iteration {iteration:2d} | Error: {error:10.8f} | Prediction: {pred:8.6f} | Weight: {weight:8.6f}")

print(f"Final Weight: {weight:.4f} (Expected: {goal_pred / input_val:.4f})")

In [ ]:
# 3.5 Demonstration of Divergence / Overshooting with Excessive Alpha

def test_learning_rate(alpha_val, iterations=6):
    w = 0.5
    inp = 2.0
    target = 0.8
    history = []
    print(f"\nTesting Alpha = {alpha_val}:")
    for i in range(iterations):
        p = inp * w
        err = (p - target) ** 2
        d = p - target
        w_delta = d * inp
        w = w - (alpha_val * w_delta)
        history.append(err)
        print(f"  Iter {i}: Error = {err:14.6f} | Pred = {p:12.6f} | W = {w:12.6f}")
    return history

# Compare standard alpha vs overshooting alpha
hist_stable = test_learning_rate(alpha_val=0.1)
hist_overshoot = test_learning_rate(alpha_val=0.6)  # Causes overshooting oscillation/explosion

---
## 4. Chapter 4 Summary & Key Takeaways

| Concept | Mathematical Equation | Key Takeaway |
|:---|:---:|:---|
| **Squared Error** | $\text{Error} = (\hat{y} - y)^2$ | Guarantees non-negative loss, heavily penalizes major outliers, and yields a smooth convex bowl. |
| **Delta (Raw Error)** | $\text{delta} = \hat{y} - y$ | Measures both the sign and magnitude of the discrepancy. |
| **Weight Delta** | $\text{weight\_delta} = \text{delta} \cdot \text{input}$ | The exact derivative $\frac{\partial \text{Error}}{\partial w}$, determining the direction and sensitivity of weight adjustment. |
| **Learning Rate ($\alpha$)** | $w \leftarrow w - \alpha \cdot \text{weight\_delta}$ | Crucial scaling factor to prevent divergence and ensure stable convergence. |
| **Next Step** | In **Chapter 5**, we generalize gradient descent to simultaneously optimize **multiple weights across multiple inputs and outputs**. |